# Demo: Vanilla RAG (Qdrant Search + LLM Answer)


Simple RAG pipeline:
1. Search Qdrant (hybrid search with RRF)
2. Build context from results
3. Generate answer with LLM

No agents, no tools - just retrieval + generation.

Prerequisites:
    1. Run data ingestion first: python demo_ingest_data.py
    2. Set up .env with QDRANT_CLOUD_URL, QDRANT_API_KEY, OPENAI_API_KEY

Usage:
    python demo_vanilla_rag.py

In [ ]:
#pip install qdrant-client[fastembed]

In [ ]:
import os
from qdrant_client import QdrantClient, models
from fastembed import TextEmbedding, SparseTextEmbedding
from openai import OpenAI
from google.colab import userdata

# Configuration
QDRANT_URL = userdata.get('QDRANT_CLOUD_URL')
QDRANT_API_KEY = userdata.get('QDRANT_API_KEY')
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
COLLECTION_NAME = "california_housing_statutes"

print("✓ Configuration loaded")

✓ Configuration loaded


## Step 1: Initialize Clients


In [ ]:
print("\n" + "="*80)
print("VANILLA RAG DEMO")
print("Qdrant Hybrid Search + LLM Generation")
print("="*80)

print("\nInitializing clients...")
client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)
dense_model = TextEmbedding(model_name="BAAI/bge-small-en-v1.5")
sparse_model = SparseTextEmbedding(model_name="Qdrant/bm42-all-minilm-l6-v2-attentions")
openai_client = OpenAI(api_key=OPENAI_API_KEY)
print("✓ Qdrant, FastEmbed, OpenAI ready\n")



VANILLA RAG DEMO
Qdrant Hybrid Search + LLM Generation

Initializing clients...
✓ Qdrant, FastEmbed, OpenAI ready



## Step 2: Define Your Question


In [ ]:
# Change this to ask your own question
question = "Can a landlord evict a tenant without providing written notice?"
print(f"Question: {question}\n")


Question: Can a landlord evict a tenant without providing written notice?



## Step 3: Retrieval - Search Qdrant (Hybrid Search)


In [ ]:
print("STEP 1: RETRIEVAL (Qdrant Hybrid Search)")
print("-" * 80)

# Generate embeddings
dense_query = list(dense_model.embed([question]))[0].tolist()
sparse_query_embedding = list(sparse_model.embed([question]))[0]
sparse_query = models.SparseVector(
    indices=sparse_query_embedding.indices.tolist(),
    values=sparse_query_embedding.values.tolist()
)

# Hybrid search with RRF
results = client.query_points(
    collection_name=COLLECTION_NAME,
    prefetch=[
        models.Prefetch(query=dense_query, using="dense", limit=20),
        models.Prefetch(query=sparse_query, using="sparse", limit=20),
    ],
    query=models.FusionQuery(fusion=models.Fusion.RRF),
    limit=5,
)

print(f"✓ Retrieved {len(results.points)} statutes\n")

# Show retrieved statutes
print("Retrieved Statutes:\n")
for i, point in enumerate(results.points, 1):
    citation = point.payload.get('citation', 'Unknown')
    text = point.payload.get('text', '')
    score = point.score

    print(f"[{i}] {citation} (score: {score:.3f})")
    print(f"    {text[:150]}...")
    print()


STEP 1: RETRIEVAL (Qdrant Hybrid Search)
--------------------------------------------------------------------------------
✓ Retrieved 5 statutes

Retrieved Statutes:

[1] CA Civ Code § 17 (2021) (score: 0.700)
    17.  Wherever any notice or other communication is required by this code to be mailed by registered mail, the mailing of such notice or other communic...

[2] CA Civ Code § 18 (2021) (score: 0.700)
    18.  Notice is:1. Actual—which consists in express information of a fact; or,2. Constructive—which is imputed by law.(Enacted 1872.)...

[3] CA Civ Code § 19 (2021) (score: 0.583)
    19.  Every person who has actual notice of circumstances sufficient to put a prudent person upon inquiry as to a particular fact has constructive noti...

[4] CA Bus & Prof Code § 11 (2021) (score: 0.389)
    11.  Writing includes any form of recorded message capable of comprehension by ordinary visual means. Whenever any notice, report, statement, or recor...

[5] CA Civ Code § 6 (2021) (score: 0

## Step 4: Generation - Build Context and Get LLM Answer


In [ ]:
print("\nSTEP 2: GENERATION (LLM Answer)")
print("-" * 80)

# Build context
context_parts = []
for i, point in enumerate(results.points, 1):
    citation = point.payload.get('citation', 'Unknown')
    text = point.payload.get('text', '')
    context_parts.append(f"[{i}] {citation}\n{text}")

context = "\n\n".join(context_parts)

# Simple prompt
prompt = f"""Based on these California housing law statutes, answer the question clearly.

Question: {question}

Statutes:
{context}

Instructions:
- Provide a clear, direct answer
- Cite specific statute sections
- Quote relevant language

Answer:"""

print("  → Building context from 5 statutes...")
print("  → Calling OpenAI GPT-4o-mini...")

# Generate answer
response = openai_client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}],
    temperature=0.3,
    max_tokens=300
)

answer = response.choices[0].message.content.strip()
print("✓ Answer generated\n")



STEP 2: GENERATION (LLM Answer)
--------------------------------------------------------------------------------
  → Building context from 5 statutes...
  → Calling OpenAI GPT-4o-mini...
✓ Answer generated



## 📝 Final Answer


In [ ]:
print("="*80)
print("ANSWER")
print("="*80)
print(f"\n{answer}\n")


ANSWER

No, a landlord cannot evict a tenant without providing written notice. According to California law, any notice required for eviction must be in writing. Specifically, CA Bus & Prof Code § 11 states, "Whenever any notice, report, statement, or record is required by this code, it shall be made in writing in the English language unless it is otherwise expressly provided." Therefore, written notice is a necessary step in the eviction process.

